### Bu projede öğrencilerin akademik durumlarının mevcut demografik, sosyoekonomik ve akademik performans verileri kullanılarak sınıflandırılması amaçlanmıştır.

In [1]:
import numpy as np 
import pandas as pd 


import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))



/kaggle/input/competitions/playground-series-s4e6/sample_submission.csv
/kaggle/input/competitions/playground-series-s4e6/train.csv
/kaggle/input/competitions/playground-series-s4e6/test.csv


In [2]:
# Veri setlerini yükleme

train_df = pd.read_csv(
    "/kaggle/input/competitions/playground-series-s4e6/train.csv"
)

test_df = pd.read_csv(
    "/kaggle/input/competitions/playground-series-s4e6/test.csv"
)

sample_submission = pd.read_csv(
    "/kaggle/input/competitions/playground-series-s4e6/sample_submission.csv"
)

print("Train Shape:", train_df.shape)
print("Test Shape:", test_df.shape)

Train Shape: (76518, 38)
Test Shape: (51012, 37)


In [3]:
# İlk gözlemler

train_df.head()

,id,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,0,1,1,1,9238,1,1,126.0,1,1,...,0,6,7,6,12.428571,0,11.1,0.6,2.02,Graduate
1,1,1,17,1,9238,1,1,125.0,1,19,...,0,6,9,0,0.000000,0,11.1,0.6,2.02,Dropout
2,2,1,17,2,9254,1,1,137.0,1,3,...,0,6,0,0,0.000000,0,16.2,0.3,-0.92,Dropout
3,3,1,1,3,9500,1,1,131.0,1,19,...,0,8,11,7,12.820000,0,11.1,0.6,2.02,Enrolled
4,4,1,1,2,9500,1,1,132.0,1,19,...,0,7,12,6,12.933333,0,7.6,2.6,0.32,Graduate


In [4]:
# Veri seti yapısını inceleme

train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76518 entries, 0 to 76517
Data columns (total 38 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   id                                              76518 non-null  int64  
 1   Marital status                                  76518 non-null  int64  
 2   Application mode                                76518 non-null  int64  
 3   Application order                               76518 non-null  int64  
 4   Course                                          76518 non-null  int64  
 5   Daytime/evening attendance                      76518 non-null  int64  
 6   Previous qualification                          76518 non-null  int64  
 7   Previous qualification (grade)                  76518 non-null  float64
 8   Nacionality                                     76518 non-null  int64  
 9   Mother's qualification                 

In [5]:
# Hedef değişken dağılımı

train_df["Target"].value_counts()

Target
Graduate    36282
Dropout     25296
Enrolled    14940
Name: count, dtype: int64

In [6]:
# Hedef değişken oranları

train_df["Target"].value_counts(normalize=True) * 100

Target
Graduate    47.416294
Dropout     33.058888
Enrolled    19.524818
Name: proportion, dtype: float64

In [7]:
# Sayısal değişkenlerin hedef ile ilişkisi

target_map = {
    "Dropout": 0,
    "Enrolled": 1,
    "Graduate": 2
}

temp_df = train_df.copy()

temp_df["target_num"] = temp_df["Target"].map(target_map)

correlations = (
    temp_df.corr(numeric_only=True)["target_num"]
    .sort_values(ascending=False)
)

correlations.head(20)

target_num                                1.000000
Curricular units 2nd sem (approved)       0.781452
Curricular units 1st sem (approved)       0.725490
Curricular units 2nd sem (grade)          0.719036
Curricular units 1st sem (grade)          0.661355
Tuition fees up to date                   0.415691
Scholarship holder                        0.394124
Curricular units 2nd sem (enrolled)       0.289165
Curricular units 1st sem (enrolled)       0.263657
Curricular units 2nd sem (evaluations)    0.214951
Admission grade                           0.172880
Course                                    0.154208
Curricular units 1st sem (evaluations)    0.152398
Displaced                                 0.150066
Previous qualification (grade)            0.138119
Application order                         0.128394
Daytime/evening attendance                0.124484
GDP                                       0.106462
Curricular units 2nd sem (credited)       0.038062
Curricular units 1st sem (credi

In [8]:
# Eksik veri kontrolü

train_df.isnull().sum().sort_values(
    ascending=False
).head(10)

id                                0
Marital status                    0
Application mode                  0
Application order                 0
Course                            0
Daytime/evening attendance        0
Previous qualification            0
Previous qualification (grade)    0
Nacionality                       0
Mother's qualification            0
dtype: int64

### Feature Engineering

In [9]:
#Bir öğrencinin genel akademik performansı.
train_df["academic_success"] = (train_df["Curricular units 1st sem (grade)"] + train_df["Curricular units 2nd sem (grade)"])

test_df["academic_success"] = (test_df["Curricular units 1st sem (grade)"] + test_df["Curricular units 2nd sem (grade)"])

#Toplam Geçilen Ders
train_df["total_approved"] = (train_df["Curricular units 1st sem (approved)"] + train_df["Curricular units 2nd sem (approved)"])

test_df["total_approved"] = (test_df["Curricular units 1st sem (approved)"] + test_df["Curricular units 2nd sem (approved)"])

#Aldığı derslerin yüzde kaçını geçti?
train_df["approval_rate"] = (train_df["total_approved"] /
    (train_df["Curricular units 1st sem (enrolled)"] + train_df["Curricular units 2nd sem (enrolled)"] + 1))

test_df["approval_rate"] = (test_df["total_approved"] / 
                            (test_df["Curricular units 1st sem (enrolled)"] + test_df["Curricular units 2nd sem (enrolled)"] + 1))

#Dönemler Arası Gelişim
train_df["performance_change"] = (train_df["Curricular units 2nd sem (grade)"] - train_df["Curricular units 1st sem (grade)"])

test_df["performance_change"] = (test_df["Curricular units 2nd sem (grade)"] - test_df["Curricular units 1st sem (grade)"])

#Akademik Verimlilik
train_df["evaluation_efficiency"] = (train_df["total_approved"] /
    (train_df["Curricular units 1st sem (evaluations)"] + train_df["Curricular units 2nd sem (evaluations)"] + 1))

test_df["evaluation_efficiency"] = (test_df["total_approved"] /
    (test_df["Curricular units 1st sem (evaluations)"] + test_df["Curricular units 2nd sem (evaluations)"] + 1))

In [10]:
# Oluşturulan feature'ların kontrolü

new_features = [
    "academic_success",
    "total_approved",
    "approval_rate",
    "performance_change",
    "evaluation_efficiency"
]

train_df[new_features].describe()

,academic_success,total_approved,approval_rate,performance_change,evaluation_efficiency
count,76518.000000,76518.000000,76518.000000,76518.000000,76518.000000
mean,19.621947,8.185721,0.602259,-0.369777,0.493979
std,10.507293,5.354344,0.367095,2.556932,0.344719
min,0.000000,0.000000,0.000000,-18.000000,0.000000
25%,20.000000,3.000000,0.272727,-0.500000,0.160000
50%,24.250000,10.000000,0.769231,0.000000,0.555556
75%,26.333333,12.000000,0.923077,0.333333,0.800000
max,36.567308,43.000000,16.000000,16.500000,15.000000


In [11]:
# Hedef değişken kodlama

target_map = {
    "Dropout": 0,
    "Enrolled": 1,
    "Graduate": 2
}

reverse_target_map = {
    0: "Dropout",
    1: "Enrolled",
    2: "Graduate"
}

y = train_df["Target"].map(target_map)

In [12]:
# Model veri seti

x = train_df.drop(columns=["id", "Target"])

test_ids = test_df["id"]

x_test = test_df.drop(columns=["id"])

In [13]:
# Kategorik değişkenler

categorical_features = x.select_dtypes(
    include=["object"]
).columns.tolist()

categorical_features

[]

In [14]:
# Train Validation ayrımı

from sklearn.model_selection import train_test_split

x_train, x_valid, y_train, y_valid = train_test_split(x, y, test_size=0.20, random_state=42, stratify=y)

In [15]:
from catboost import CatBoostClassifier

model = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.03,
    depth=8,
    loss_function="MultiClass",
    eval_metric="MultiClass",
    random_state=42,
    verbose=200
)

model.fit(
    x_train,
    y_train,
    eval_set=(x_valid, y_valid),
    use_best_model=True
)

0:	learn: 1.0646258	test: 1.0647615	best: 1.0647615 (0)	total: 118ms	remaining: 3m 55s
200:	learn: 0.4417184	test: 0.4551793	best: 0.4551793 (200)	total: 9.5s	remaining: 1m 25s
400:	learn: 0.4190450	test: 0.4453956	best: 0.4453956 (400)	total: 19s	remaining: 1m 15s
600:	learn: 0.4017484	test: 0.4413262	best: 0.4413118 (599)	total: 28.5s	remaining: 1m 6s
800:	learn: 0.3878240	test: 0.4394433	best: 0.4394410 (798)	total: 38s	remaining: 56.9s
1000:	learn: 0.3757948	test: 0.4382907	best: 0.4382657 (996)	total: 47.5s	remaining: 47.4s
1200:	learn: 0.3641711	test: 0.4372529	best: 0.4372452 (1196)	total: 56.9s	remaining: 37.9s
1400:	learn: 0.3542271	test: 0.4366122	best: 0.4366122 (1400)	total: 1m 6s	remaining: 28.3s
1600:	learn: 0.3438733	test: 0.4364334	best: 0.4363501 (1581)	total: 1m 15s	remaining: 18.9s
1800:	learn: 0.3342818	test: 0.4363575	best: 0.4362901 (1705)	total: 1m 25s	remaining: 9.41s
1999:	learn: 0.3249389	test: 0.4366825	best: 0.4362901 (1705)	total: 1m 34s	remaining: 0us

bes

CatBoostClassifier(depth=8, eval_metric='MultiClass', iterations=2000, learning_rate=0.03, loss_function='MultiClass', random_state=42, verbose=200)

In [16]:
# Validation tahminleri

valid_preds = model.predict(x_valid)

In [17]:
# Accuracy skoru

from sklearn.metrics import accuracy_score

accuracy = accuracy_score(
    y_valid,
    valid_preds
)

print("Validation Accuracy:", accuracy)

Validation Accuracy: 0.8308285415577626


In [18]:
# Macro F1 skoru

from sklearn.metrics import f1_score

macro_f1 = f1_score(
    y_valid,
    valid_preds,
    average="macro"
)

print("Validation Macro F1:", macro_f1)

Validation Macro F1: 0.7915562065353409


In [19]:
# Feature importance

import pandas as pd

importance_df = pd.DataFrame({
    "Feature": x.columns,
    "Importance": model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

importance_df.head(20)

,Feature,Importance
38,approval_rate,7.439957
30,Curricular units 2nd sem (approved),6.622751
3,Course,5.734841
12,Admission grade,5.495738
40,evaluation_efficiency,5.202773
31,Curricular units 2nd sem (grade),5.153598
29,Curricular units 2nd sem (evaluations),4.492656
6,Previous qualification (grade),3.990812
16,Tuition fees up to date,3.978343
36,academic_success,3.868589


In [20]:
# Test verisi için tahmin oluşturma

test_predictions = model.predict(x_test)

test_predictions = [
    reverse_target_map[int(pred[0])]
    for pred in test_predictions
]

submission = pd.DataFrame({
    "id": test_ids,
    "Target": test_predictions
})

submission.to_csv(
    "submission.csv",
    index=False
)

submission.head()

,id,Target
0,76518,Dropout
1,76519,Graduate
2,76520,Graduate
3,76521,Graduate
4,76522,Enrolled


### CatBoost modeli ile yapılan çalışmada Kaggle yarışmasında 0.83258 public score ve 0.83241 private score elde edilmiştir. Feature importance sonuçlarına göre approval_rate, Curricular units 2nd sem (approved), Course, Admission grade ve evaluation_efficiency değişkenleri model kararlarında öne çıkmıştır. Sonuçlar, öğrencinin akademik başarısının özellikle dönem içi ders geçme oranı, not performansı ve eğitim sürecindeki devamlılık göstergeleriyle güçlü biçimde ilişkili olduğunu göstermektedir.

In [21]:
# Model ve gerekli dosyaları kaydetme

import joblib

model.save_model("academic_success_model.cbm")

feature_columns = x_train.columns.tolist()

joblib.dump(
    feature_columns,
    "feature_columns.pkl"
)

joblib.dump(
    target_map,
    "target_map.pkl"
)

joblib.dump(
    reverse_target_map,
    "reverse_target_map.pkl"
)

['reverse_target_map.pkl']